# ERP Connect CogitION v3.0 - TinyLlama Training Pipeline

Complete AI training pipeline for CTRM-SAP integration support system.

**Model:** TinyLlama-1.1B-Chat-v1.0  
**Training Method:** LoRA Fine-tuning  
**Features:** RAG-enhanced context retrieval  
**Hardware:** Google Colab T4 GPU  

**Training Time:** Approximately 35-40 minutes  
**Persistence:** All files and models saved to Google Drive

## Cell 1: Install Required Packages

In [ ]:
print("Installing required packages...\n")

!pip install -q transformers datasets peft accelerate sentencepiece
!pip install -q rouge-score nltk sentence-transformers scikit-learn
!pip install -q faiss-cpu gradio

print("\nPackages installed successfully.\n")

import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected. Please enable GPU in Runtime > Change runtime type.")

## Cell 1A: Mount Google Drive and Setup Folders

In [ ]:
print("="*70)
print("MOUNTING GOOGLE DRIVE AND SETTING UP FOLDER STRUCTURE")
print("="*70)

from google.colab import drive
import os
import shutil

print("\nMounting Google Drive...")
drive.mount('/content/drive')
print("Drive mounted successfully.")

base_path = '/content/drive/MyDrive/ERP_CogitION'
data_path = os.path.join(base_path, 'data')
models_path = os.path.join(base_path, 'models')
trained_path = os.path.join(base_path, 'trained_model')

print("\nCreating folder structure...")
os.makedirs(data_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)
os.makedirs(trained_path, exist_ok=True)
print("Folders created successfully.")

print("\nGoogle Drive folder structure:")
print(f"  Base: {base_path}")
print(f"  Data: {data_path}")
print(f"  Models: {models_path}")
print(f"  Trained Model: {trained_path}")

print("\n" + "="*70)
print("GOOGLE DRIVE SETUP COMPLETE")
print("="*70)
print("\nFiles saved to Google Drive will persist across sessions.")

## Cell 2A: Upload Files to Google Drive (One-Time Setup)

**Use this cell on first run only.**  
Upload the following 5 files:
- train.jsonl
- validation.jsonl
- test.jsonl
- knowledge_base_v2.index
- documents_v2.pkl

In [ ]:
print("="*70)
print("UPLOAD FILES TO GOOGLE DRIVE (ONE-TIME SETUP)")
print("="*70)

from google.colab import files
import shutil

base_path = '/content/drive/MyDrive/ERP_CogitION'
data_path = os.path.join(base_path, 'data')
models_path = os.path.join(base_path, 'models')

print("\nPlease upload the following 5 files:")
print("  1. train.jsonl")
print("  2. validation.jsonl")
print("  3. test.jsonl")
print("  4. knowledge_base_v2.index")
print("  5. documents_v2.pkl")
print("\nClick 'Choose Files' button below...\n")

uploaded = files.upload()

print("\nCopying files to Google Drive...")

for filename in uploaded.keys():
    if filename.endswith('.jsonl'):
        dest = os.path.join(data_path, filename)
        shutil.copy(filename, dest)
        print(f"  Saved: {filename} to data/")
    elif filename.endswith(('.index', '.pkl')):
        dest = os.path.join(models_path, filename)
        shutil.copy(filename, dest)
        print(f"  Saved: {filename} to models/")

print("\n" + "="*70)
print("FILES SAVED TO GOOGLE DRIVE SUCCESSFULLY")
print("="*70)
print("\nNote: You only need to run this cell once.")
print("For subsequent sessions, use Cell 2B to load files from Drive.")

## Cell 2B: Load Files from Google Drive (Fast Load)

**Use this cell for all subsequent runs.**  
Loads previously uploaded files from Google Drive to working directory.

In [ ]:
print("="*70)
print("LOADING FILES FROM GOOGLE DRIVE")
print("="*70)

import shutil
import os

base_path = '/content/drive/MyDrive/ERP_CogitION'
data_path = os.path.join(base_path, 'data')
models_path = os.path.join(base_path, 'models')

print("\nCopying data files to working directory...")

data_files = ['train.jsonl', 'validation.jsonl', 'test.jsonl']
for filename in data_files:
    src = os.path.join(data_path, filename)
    if os.path.exists(src):
        shutil.copy(src, f'/content/{filename}')
        size_kb = os.path.getsize(src) / 1024
        print(f"  Loaded: {filename} ({size_kb:.1f} KB)")
    else:
        print(f"  ERROR: {filename} not found in Google Drive!")

print("\nCopying model files to working directory...")

model_files = ['knowledge_base_v2.index', 'documents_v2.pkl']
for filename in model_files:
    src = os.path.join(models_path, filename)
    if os.path.exists(src):
        shutil.copy(src, f'/content/{filename}')
        size_kb = os.path.getsize(src) / 1024
        print(f"  Loaded: {filename} ({size_kb:.1f} KB)")
    else:
        print(f"  ERROR: {filename} not found in Google Drive!")

print("\n" + "="*70)
print("FILES LOADED FROM GOOGLE DRIVE SUCCESSFULLY")
print("="*70)

## Cell 3: Verify Files

In [ ]:
print("="*70)
print("FILE VERIFICATION")
print("="*70)

import os
import json

files_to_check = {
    'train.jsonl': 1200,
    'validation.jsonl': 150,
    'test.jsonl': 150,
    'knowledge_base_v2.index': None,
    'documents_v2.pkl': None
}

all_present = True

for filename, expected_samples in files_to_check.items():
    if os.path.exists(filename):
        size_kb = os.path.getsize(filename) / 1024
        print(f"\nFile: {filename} ({size_kb:.1f} KB)")
        
        if filename.endswith('.jsonl'):
            with open(filename, 'r') as f:
                lines = f.readlines()
                actual_samples = len(lines)
                print(f"  Samples: {actual_samples}")
                
                if actual_samples == expected_samples:
                    print(f"  Status: Verified (expected {expected_samples})")
                    sample = json.loads(lines[0])
                    print(f"  Sample prompt: {sample['prompt'][:60]}...")
                else:
                    print(f"  WARNING: Expected {expected_samples} samples, found {actual_samples}")
                    all_present = False
        else:
            print(f"  Status: Present")
    else:
        print(f"\nFile: {filename}")
        print(f"  Status: MISSING")
        all_present = False

print("\n" + "="*70)
if all_present:
    print("ALL FILES VERIFIED SUCCESSFULLY")
    print("Ready to proceed with training.")
else:
    print("SOME FILES ARE MISSING OR INCORRECT")
    print("Please upload missing files before proceeding.")
print("="*70)

## Cell 4: Train TinyLlama with LoRA

**Training Configuration:**
- Model: TinyLlama-1.1B-Chat-v1.0
- Method: LoRA fine-tuning (r=16, alpha=32)
- Epochs: 3
- Batch size: 4 with gradient accumulation 4 (effective batch size 16)
- Learning rate: 2e-4
- Precision: FP16

**Expected Training Time:** 35-40 minutes  
**Target Final Loss:** 0.8-1.2  
**Model will be automatically saved to Google Drive**

In [ ]:
print("="*70)
print("TRAINING TINYLLAMA-1.1B WITH LORA")
print("="*70)

from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch
import shutil

print("\nLoading TinyLlama-1.1B model...")
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)
print("Model loaded successfully.")

print("\nConfiguring LoRA adapters...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("LoRA adapters added successfully.")

print("\nLoading training datasets...")
train_dataset = load_dataset('json', data_files='train.jsonl', split='train')
val_dataset = load_dataset('json', data_files='validation.jsonl', split='train')
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

def preprocess_function(examples):
    texts = []
    for prompt, response in zip(examples['prompt'], examples['response']):
        text = f"<|user|>\n{prompt}</s>\n<|assistant|>\n{response}</s>"
        texts.append(text)
    
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("\nTokenizing datasets...")
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=['prompt', 'response'])
val_dataset = val_dataset.map(preprocess_function, batched=True, remove_columns=['prompt', 'response'])
print("Tokenization complete.")

print("\nVerifying sample preprocessing...")
sample = train_dataset[0]
decoded = tokenizer.decode(sample['input_ids'][:150])
print(f"Sample preview: {decoded[:100]}...")

training_args = TrainingArguments(
    output_dir="tinyllama-fine-tuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    logging_steps=20,
    eval_steps=75,
    save_steps=75,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=2,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)
print("\nExpected first loss: 1.5-2.5")
print("Target final loss: 0.8-1.2")
print("Estimated time: 35-40 minutes")
print("\nMonitor the loss values below...\n")

trainer.train()

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)

print("\nSaving model to local directory...")
trainer.save_model("tinyllama-fine-tuned")
tokenizer.save_pretrained("tinyllama-fine-tuned")
print("Model saved locally.")

print("\nCopying model to Google Drive...")
drive_model_path = '/content/drive/MyDrive/ERP_CogitION/trained_model'
if os.path.exists(drive_model_path):
    shutil.rmtree(drive_model_path)
shutil.copytree("tinyllama-fine-tuned", drive_model_path)
print(f"Model saved to: {drive_model_path}")

print("\n" + "="*70)
print("MODEL SAVED SUCCESSFULLY")
print("="*70)
print("\nThe trained model is now persisted in Google Drive.")
print("It will be available in future sessions without retraining.")

## Cell 5: Evaluate Model Performance

Evaluates the fine-tuned model on test dataset using:
- ROUGE scores (ROUGE-1, ROUGE-2, ROUGE-L)
- BLEU score
- Cosine similarity

**Evaluation Time:** Approximately 30-35 minutes for 150 samples

In [ ]:
print("="*70)
print("EVALUATING TINYLLAMA MODEL")
print("="*70)

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from peft import PeftModel
import torch

print("\nLoading test dataset...")
test_dataset = load_dataset('json', data_files='test.jsonl', split='train')
print(f"Test samples: {len(test_dataset)}")

print("\nLoading trained model...")
base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto",
    torch_dtype=torch.float16
)

drive_model_path = '/content/drive/MyDrive/ERP_CogitION/trained_model'
if os.path.exists(drive_model_path):
    print("Loading model from Google Drive...")
    model = PeftModel.from_pretrained(base_model, drive_model_path)
else:
    print("Loading model from local directory...")
    model = PeftModel.from_pretrained(base_model, "tinyllama-fine-tuned")

model.eval()
print("Model loaded successfully.")

print("\nInitializing evaluation metrics...")
rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
similarity_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Metrics initialized.")

def generate_answer(prompt):
    input_text = f"<|user|>\n{prompt}</s>\n<|assistant|>\n"
    inputs = tokenizer(input_text, return_tensors='pt', truncation=True, max_length=256).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )
    
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = answer.split("<|assistant|>")[-1].strip()
    
    if len(answer) > 1000:
        sentences = answer.split('. ')
        answer = '. '.join(sentences[:-1]) + '.'
    
    return answer

print("\nEvaluating model performance...")
print("This will take approximately 30-35 minutes.\n")

results = {'rouge1': [], 'rouge2': [], 'rougeL': [], 'bleu': [], 'cosine': []}
samples_to_show = []

for idx, sample in enumerate(test_dataset):
    prediction = generate_answer(sample['prompt'])
    reference = sample['response']
    
    rouge_scores = rouge_scorer_obj.score(reference, prediction)
    results['rouge1'].append(rouge_scores['rouge1'].fmeasure)
    results['rouge2'].append(rouge_scores['rouge2'].fmeasure)
    results['rougeL'].append(rouge_scores['rougeL'].fmeasure)
    
    ref_tokens = reference.split()
    pred_tokens = prediction.split()
    smoothing = SmoothingFunction().method1
    bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothing)
    results['bleu'].append(bleu)
    
    ref_emb = similarity_model.encode([reference])
    pred_emb = similarity_model.encode([prediction])
    cosine = cosine_similarity(ref_emb, pred_emb)[0][0]
    results['cosine'].append(float(cosine))
    
    if idx < 5:
        samples_to_show.append({
            'prompt': sample['prompt'],
            'prediction': prediction,
            'reference': reference[:100] + '...'
        })
    
    if (idx + 1) % 30 == 0:
        print(f"Progress: {idx + 1}/{len(test_dataset)}")

print("\n" + "="*70)
print("EVALUATION RESULTS")
print("="*70)

metrics = {
    'ROUGE-1': results['rouge1'],
    'ROUGE-2': results['rouge2'],
    'ROUGE-L': results['rougeL'],
    'BLEU': results['bleu'],
    'Cosine Similarity': results['cosine']
}

for name, values in metrics.items():
    mean_val = np.mean(values)
    print(f"\n{name}:")
    print(f"  Mean:  {mean_val:.4f}")
    print(f"  Std:   {np.std(values):.4f}")
    print(f"  Min:   {np.min(values):.4f}")
    print(f"  Max:   {np.max(values):.4f}")

print("\n" + "="*70)
print("TARGET COMPARISON")
print("="*70)

rouge_l = np.mean(results['rougeL'])
bleu = np.mean(results['bleu'])
cosine = np.mean(results['cosine'])

print(f"\nMetric          Target    Actual    Status")
print(f"{'='*45}")
print(f"ROUGE-L         0.75      {rouge_l:.4f}    {'PASS' if rouge_l >= 0.75 else 'CLOSE' if rouge_l >= 0.70 else 'FAIL'}")
print(f"BLEU            0.65      {bleu:.4f}    {'PASS' if bleu >= 0.65 else 'CLOSE' if bleu >= 0.60 else 'FAIL'}")
print(f"Cosine Sim      0.90      {cosine:.4f}    {'PASS' if cosine >= 0.90 else 'CLOSE' if cosine >= 0.85 else 'FAIL'}")

print("\n" + "="*70)
print("SAMPLE PREDICTIONS")
print("="*70)

for i, sample in enumerate(samples_to_show, 1):
    print(f"\nSample {i}:")
    print(f"Question: {sample['prompt']}")
    print(f"\nAnswer: {sample['prediction']}")
    print(f"\nExpected: {sample['reference']}")
    print("-" * 70)

print("\nEvaluation complete.")

## Cell 6: Launch Gradio Demo

Creates an interactive web interface for testing the model.

**Features:**
- Question input with RAG context retrieval
- Adjustable temperature parameter
- Source attribution
- Public shareable URL

The demo will generate a public URL that can be shared with others.

In [ ]:
print("="*70)
print("LAUNCHING GRADIO DEMO")
print("="*70)

import gradio as gr
import pickle
import faiss

print("\nLoading model components...")

base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto",
    torch_dtype=torch.float16
)

drive_model_path = '/content/drive/MyDrive/ERP_CogitION/trained_model'
if os.path.exists(drive_model_path):
    model = PeftModel.from_pretrained(base_model, drive_model_path)
else:
    model = PeftModel.from_pretrained(base_model, "tinyllama-fine-tuned")

model.eval()
print("Model loaded.")

index = faiss.read_index('knowledge_base_v2.index')
with open('documents_v2.pkl', 'rb') as f:
    documents = pickle.load(f)
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(f"RAG system loaded with {len(documents)} document chunks.")

def retrieve_context(query, top_k=3):
    query_embedding = embedder.encode([query], normalize_embeddings=True)
    distances, indices = index.search(query_embedding.astype('float32'), top_k)
    
    contexts = []
    sources = []
    for idx in indices[0]:
        if idx < len(documents):
            doc = documents[idx]
            contexts.append(doc['text'][:200])
            sources.append(doc.get('source_file', 'Unknown'))
    
    return "\n\n".join(contexts), sources

def answer_question(question, use_rag=True, temperature=0.7):
    if not question.strip():
        return "Please enter a question.", "", ""
    
    context_text = ""
    sources_text = ""
    
    if use_rag:
        context, sources = retrieve_context(question)
        context_text = context
        sources_text = "\n".join([f"- {s}" for s in set(sources)])
        prompt = f"Context:\n{context}\n\nQuestion: {question}"
    else:
        prompt = question
    
    input_text = f"<|user|>\n{prompt}</s>\n<|assistant|>\n"
    inputs = tokenizer(input_text, return_tensors='pt', truncation=True, max_length=256).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            max_new_tokens=300,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )
    
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = answer.split("<|assistant|>")[-1].strip()
    
    if len(answer) > 1000:
        sentences = answer.split('. ')
        answer = '. '.join(sentences[:-1]) + '.'
    
    return answer, context_text, sources_text

print("\nBuilding interface...")

with gr.Blocks(title="ERP Connect CogitION - TinyLlama") as demo:
    gr.Markdown("""
    # ERP Connect CogitION - TinyLlama Edition
    
    AI-powered CTRM-SAP integration support system
    
    **Model:** TinyLlama-1.1B fine-tuned with LoRA  
    **Features:** RAG-enhanced context retrieval
    """)
    
    with gr.Row():
        with gr.Column(scale=2):
            question_input = gr.Textbox(
                label="Your Question",
                placeholder="Ask about CTRM-SAP integration...",
                lines=3
            )
            
            with gr.Row():
                submit_btn = gr.Button("Get Answer", variant="primary")
                clear_btn = gr.Button("Clear")
            
            with gr.Accordion("Advanced Options", open=False):
                use_rag = gr.Checkbox(label="Use RAG", value=True)
                temperature = gr.Slider(0.1, 1.0, value=0.7, label="Temperature")
        
        with gr.Column(scale=3):
            answer_output = gr.Textbox(label="Answer", lines=10)
            
            with gr.Accordion("Retrieved Context", open=False):
                context_output = gr.Textbox(label="Context Used", lines=5)
            
            with gr.Accordion("Sources", open=False):
                sources_output = gr.Textbox(label="Sources", lines=3)
    
    gr.Examples(
        examples=[
            ["How to add interface mapping for Cost Center ENERGY572?"],
            ["Error: Server responded with HTTP 403 Forbidden"],
            ["ECS Fargate container won't start - exit code 137"],
            ["RDS database connection timeout"],
            ["Lambda function timeout after 30 seconds"]
        ],
        inputs=question_input
    )
    
    submit_btn.click(
        fn=answer_question,
        inputs=[question_input, use_rag, temperature],
        outputs=[answer_output, context_output, sources_output]
    )
    
    clear_btn.click(
        fn=lambda: ("", "", "", ""),
        outputs=[question_input, answer_output, context_output, sources_output]
    )

print("Interface ready.")
print("\nLaunching demo...")
print("A public URL will be generated that you can share.\n")

demo.launch(share=True, debug=False)

## Cell 7: Download Trained Model

Downloads the trained model as a ZIP file for local use.

**Contents:**
- LoRA adapter weights
- Tokenizer configuration
- Model configuration files

**Size:** Approximately 50-100 MB

In [ ]:
print("="*70)
print("DOWNLOAD TRAINED MODEL")
print("="*70)

import shutil
import os
from google.colab import files

print("\nCreating ZIP archive...")
print("This may take 2-3 minutes.\n")

model_dir = 'tinyllama-fine-tuned'
if not os.path.exists(model_dir):
    drive_model_path = '/content/drive/MyDrive/ERP_CogitION/trained_model'
    if os.path.exists(drive_model_path):
        print("Copying model from Google Drive...")
        shutil.copytree(drive_model_path, model_dir)
    else:
        print("ERROR: Model not found. Please train the model first.")
        raise FileNotFoundError("Model directory not found")

shutil.make_archive('tinyllama-fine-tuned', 'zip', model_dir)

file_size = os.path.getsize('tinyllama-fine-tuned.zip') / (1024 * 1024)
print(f"ZIP file created: {file_size:.1f} MB")

print("\nStarting download...")
print("Check your browser's download folder.\n")

files.download('tinyllama-fine-tuned.zip')

print("\n" + "="*70)
print("DOWNLOAD COMPLETE")
print("="*70)

print("\nModel contents:")
print("  - LoRA adapter weights (adapter_model.safetensors)")
print("  - Adapter configuration (adapter_config.json)")
print("  - Tokenizer files")

print("\nTo use on your local machine:")
print("  1. Extract the ZIP file")
print("  2. Install required packages:")
print("     pip install transformers peft torch")
print("  3. Load the model:")
print("     from transformers import AutoModelForCausalLM, AutoTokenizer")
print("     from peft import PeftModel")
print("     base_model = AutoModelForCausalLM.from_pretrained('TinyLlama/TinyLlama-1.1B-Chat-v1.0')")
print("     model = PeftModel.from_pretrained(base_model, './tinyllama-fine-tuned')")

print("\nDownload successful.")